# Dependencies

In [73]:
from datasets import load_dataset
from dotenv import load_dotenv
from typing import Callable, Any
from pathlib import Path
import numpy as np
import torch as T
import string
import os

# Setup

In [74]:
# Change print width
T.set_printoptions(linewidth=np.inf)
np.set_printoptions(linewidth=np.inf)

# Data
Importing the data

In [75]:
load_dotenv("./.env.secrets")

hf_token = os.getenv("HF_TOKEN")

FORMAT = "torch"

ds = load_dataset(
    "Salesforce/wikitext",
    "wikitext-103-v1",
    token=hf_token if hf_token else False,
).with_format(FORMAT)

x_test, x_train, x_val = (ds[key] for key in ds.keys())

display(x_train["text"][:10])

['',
 ' = Valkyria Chronicles III = \n',
 '',
 ' Senjō no Valkyria 3 : <unk> Chronicles ( Japanese : 戦場のヴァルキュリア3 , lit . Valkyria of the Battlefield 3 ) , commonly referred to as Valkyria Chronicles III outside Japan , is a tactical role @-@ playing video game developed by Sega and Media.Vision for the PlayStation Portable . Released in January 2011 in Japan , it is the third game in the Valkyria series . Employing the same fusion of tactical and real @-@ time gameplay as its predecessors , the story runs parallel to the first game and follows the " Nameless " , a penal military unit serving the nation of Gallia during the Second Europan War who perform secret black operations and are pitted against the Imperial unit " <unk> Raven " . \n',
 " The game began development in 2010 , carrying over a large portion of the work done on Valkyria Chronicles II . While it retained the standard features of the series , it also underwent multiple adjustments , such as making the game more forgiving

## Data pre-processing
Preprocessing the data allows us to collect as much useful information as possible, these following operations are performed:

- Remove punctuations: Since our aim is to analyze the semantic meaning of individual words and not sentences. Including punctuation, which carries meaning in a sentences, would be misleading since we would lead to situations where two different words (for example "apple." and "apple") have slightly different meanings when they, when taken out of a sentence context, should have practically the same meaning. This is also computationally inficient if you consider that our task is strictly confined to analyzing the semantic difference and meaning of words.

- Force lower: We do this for exactly the same reasons as why we remove punctuation.

You could make the case that the words "Apple", "apple", "apple." actually do have differenet semanting meanings and technically you would be correct, but as stated above, we want to focus the model to learn the difference between "apple" and "pear" rather than using its parameters to differentiate between "Apple" and "apple".

In [76]:
def pre_process_data(data: T.utils.data.Dataset):
    # We clean the words in the dataset a bit, this is to ensure that all words present
    # in the vocabulary are in a predictable format
    to_remove = string.punctuation + "“”‘’"
    table = str.maketrans("", "", to_remove)  # Create a translation table
    big_blob = " ".join(data["text"])
    # Translate the the blob using the table and convert to lower
    cleaned_blob = big_blob.translate(table).lower()
    return sorted(list(set(cleaned_blob.split())))

# One-hot-encoding model
Assigns each word in a vocabulary a $n$ dimensional vector with a single element set to one (uniquely), the words are sorted alphabetically.

In [77]:
class OneHotEncoder:
    def __init__(self, vocabulary: list[str]):
        self.vocabulary = vocabulary
        self.dim = len(vocabulary)
        self.word_indexes = {word: i for i, word in enumerate(vocabulary)}

    def __call__(self, v: str):
        vec = T.zeros(self.dim)
        idx = self.word_indexes.get(v)
        if idx:
            vec[idx] = 1
        return vec

In [78]:
FN = "one_hot_encoder.pt"
file_path = Path(FN)
if file_path.exists():
   encoder = T.load(file_path, weights_only=False)
else:
    vocabulary = pre_process_data(x_train)
    encoder = OneHotEncoder(vocabulary)
    T.save(encoder, file_path)

# Creating the modeling tools
We choose to define our model in an object oriented way. The reason for this is that we hope to increase readability of the code mainly by relating python objects to mathematical objects, hopefully creating a successfull bridge between the theory and the implementation.

Each of layer of our model can be modeled as a function.
$$
        \mathbf{\hat{y}} = f(\mathbf{v})
$$
This function can 

In [79]:
class Derivable():

    def __init__(self):

        self.dydv: T.Tensor = None
        """
        Derivative (D) of the output y (D) w.r.t to the Input (DV) of the object.
        """

        

In [80]:
class Trainable(Derivable):

    def __init__(self):

        self.dydw: T.Tensor = None
        """
        Derivative (D) of the output y (Y) w.r.t to the Weights (DW) of the object.
        """

    def update_weights(self, dnydv: T.Tensor) -> None:
        """
        Update the weights of the object using the derivative of the output of the
        next layer w.r.t its input which is the current layer output.
        """
        raise NotImplementedError(
            f"{self.__name__} has not implemented update_weights"
        )

## No activation
This is a activation function that does not modify the data, its needed for the projection layer of Word2Vec.

In [81]:
class NoActivation(Derivable, Callable[[T.Tensor], T.Tensor]):
    def __call__(self, z: T.Tensor) -> T.Tensor:
        self.dydv = T.eye(z)
        return z

## Softmax
Softmax is a function that turns a vector of numbers $\mathbf{z} \in \mathbf{R}^n$ in to a vector of proabilities $\mathbf{\hat{y}}$ s.t $\sum_i \hat{y}_i = 1$. The softmax function is defined as:

$$\mathbf{\hat{y}}_i = \frac{e^{z_i}}{\sum_{j=1}^{n} e^{z_j}}$$

The gradients are calculated automatically during the forward pass and are set follows:

$$
    \begin{align*}
        & \delta_{ij} = \begin{cases} 1 & \text{if } i = j \\ 0 & \text{if } i \neq j \end{cases} \\
        & \frac{\partial \hat{y}_i}{\partial z_j} = \hat{y}_i(\delta_{ij} - \hat{y}_j) \\
        & \text{diag}(\mathbf{\hat{y}}) = \begin{bmatrix} \hat{y}_1 & 0 & \dots \\ 0 & \hat{y}_2 & \dots \\ \vdots & \vdots & \ddots \end{bmatrix}
        & \mathbf{\hat{y}}\mathbf{\hat{y}}^T = \begin{bmatrix} \hat{y}_1^2 & \hat{y}_1\hat{y}_2 & \dots \\ \hat{y}_2\hat{y}_1 & \hat{y}_2^2 & \dots \\ \vdots & \vdots & \ddots \end{bmatrix}\\
        & \frac{\partial{\mathbf{\hat{y}}}}{\partial{\mathbf{z}}} = \text{diag}(\mathbf{\hat{y}}) - \mathbf{\hat{y}}\mathbf{\hat{y}}^T
    \end{align*}
$$

The single element version $\frac{\partial \hat{y}_i}{\partial z_j}$ of the derivative is included for clarity as it makes it easier to understand the full vector ($\mathbf{\hat{y}}$). 

In [82]:
class Softmax(Derivable, Callable[[T.Tensor], T.Tensor]):
    def __call__(self, z: T.Tensor) -> T.Tensor:
        # Exponentiate all elements of z, z are the logits, or the "raw" input.
        z_exp = T.exp(z)
        # Squeeze away the last element of y, this makes no difference mathematically.
        y = (z_exp / T.sum(z_exp, dim=0)).squeeze()
        # Store the partial derivative dy/dz.
        self.dydv = T.diag(y) - T.outer(y.T,y)
        return y


sm = Softmax()
test_values = [
    T.tensor([1.0, 0.0, 0.0, 0.0]).reshape(-1, 1),
    T.tensor([1.0, 1.0, 0.0, 0.0]).reshape(-1, 1),
    T.tensor([1.0, 1.0, 1.0, 0.0]).reshape(-1, 1),
    T.tensor([1.0, 1.0, 1.0, 1.0]).reshape(-1, 1),
]

for v in test_values:
    out = sm(v)
    dydv = sm.dydv
    print(
        f"""Output:\n{out.numpy()},
        \nPartial derivative:\n{dydv.numpy()}
        """
    )

Output:
[0.4753669 0.1748777 0.1748777 0.1748777],
        
Partial derivative:
[[ 0.24939321 -0.08313107 -0.08313107 -0.08313107]
 [-0.08313107  0.14429548 -0.03058221 -0.03058221]
 [-0.08313107 -0.03058221  0.14429548 -0.03058221]
 [-0.08313107 -0.03058221 -0.03058221  0.14429548]]
        
Output:
[0.3655293  0.3655293  0.13447072 0.13447072],
        
Partial derivative:
[[ 0.23191763 -0.13361166 -0.04915299 -0.04915299]
 [-0.13361166  0.23191763 -0.04915299 -0.04915299]
 [-0.04915299 -0.04915299  0.11638834 -0.01808237]
 [-0.04915299 -0.04915299 -0.01808237  0.11638834]]
        
Output:
[0.29692274 0.29692274 0.29692274 0.10923178],
        
Partial derivative:
[[ 0.20875964 -0.08816312 -0.08816312 -0.0324334 ]
 [-0.08816312  0.20875964 -0.08816312 -0.0324334 ]
 [-0.08816312 -0.08816312  0.20875964 -0.0324334 ]
 [-0.0324334  -0.0324334  -0.0324334   0.09730019]]
        
Output:
[0.25 0.25 0.25 0.25],
        
Partial derivative:
[[ 0.1875 -0.0625 -0.0625 -0.0625]
 [-0.0625  0.18

## Linear layer
The object `LinearLayer2D` performs the following operation on an input vector $\mathbf{v}$:

$$
    \begin{align*}
        & \mathbf{z} = \mathbf{v}^T \mathbf{W} + \mathbf{b} \\
        & \mathbf{\hat{y}} = \sigma{(\mathbf{z})}
    \end{align*}
$$

Where $\mathbf{b}$ is the bias term, $\mathbf{W}$ is the weights, and $\mathbf{\sigma}$ is the activation function, $\mathbf{z}$ is just an intermediary step used for convenience.

We have, however, modified this operation slightly so that we can remove the separate bias term and include it directly in an expanded weight matrix $\mathbf{W_b}$. This is achieved by adding a row filled with ones to $\mathbf{W}$. So the operation actually becomes:

$$
    \begin{align*}
        & \mathbf{z} = \mathbf{v}^T \mathbf{W_b} \\
        & \mathbf{\hat{y}} = \sigma{(\mathbf{z})}
    \end{align*}
$$

The gradients are calculated as following during the forward pass:

**$\mathbf{\hat{y}}$ w.r.t the input vector $\mathbf{v}$**
$$
    \begin{align*}
        & z_i = v_1 \mathbf{W_b}_{1i} + v_2 \mathbf{W_b}_{2i} + \dots + v_i \mathbf{W_b}_{ii} + \dots + v_N \mathbf{W_b}_{ni} \\

        & \frac{\partial \mathbf{z_i}}{\partial \mathbf{v_j}} = \mathbf{W_b}_{ij} \\

        & \frac{\partial \mathbf{z}}{\partial \mathbf{v}} = \begin{bmatrix} \frac{\partial z_1}{\partial v_1} & \frac{\partial z_1}{\partial v_2} & \dots & \frac{\partial z_1}{\partial v_N} \\ \frac{\partial z_2}{\partial v_1} & \frac{\partial z_2}{\partial v_2} & \dots & \frac{\partial z_2}{\partial v_N} \\ \vdots & \vdots & \ddots & \vdots \\ \frac{\partial z_N}{\partial v_1} & \frac{\partial z_N}{\partial v_2} & \dots & \frac{\partial z_N}{\partial v_N} \end{bmatrix} = \mathbf{W_b} \\

        & \frac{\partial \mathbf{\hat{y}}}{\partial \mathbf{z}} \text{ is taken from the activation function} \\

        & \frac{\partial \mathbf{\hat{y}}}{\partial \mathbf{v}} = \frac{\partial \mathbf{\hat{y}}}{\partial \mathbf{z}} \cdot \mathbf{W_b}        
    \end{align*}
$$

**$\mathbf{\hat{y}}$ w.r.t the weight matrix $\mathbf{W}$**
$$
    \begin{align*}
        & z_i = v_1 \mathbf{W_b}_{1i} + v_2 \mathbf{W_b}_{2i} + \dots + v_i \mathbf{W_b}_{ii} + \dots + v_N \mathbf{W_b}_{ni} \\

        & \frac{\partial z_i}{\partial \mathbf{W_b}_{kj}} = \begin{cases} v_k & \text{if } i = j \\ 0 & \text{if } i \neq j \end{cases} \\

        & \frac{\partial z}{\partial W} = \left[ \quad \begin{bmatrix} v_1 & 0 & \dots & 0 \\ v_2 & 0 & \dots & 0 \\ \vdots & \vdots & \ddots & \vdots \\ v_N & 0 & \dots & 0 \end{bmatrix} \quad \Bigg| \quad \begin{bmatrix} 0 & v_1 & \dots & 0 \\ 0 & v_2 & \dots & 0 \\ \vdots & \vdots & \ddots & \vdots \\ 0 & v_N & \dots & 0 \end{bmatrix} \quad \Bigg| \quad \dots \quad \Bigg| \quad \begin{bmatrix} 0 & 0 & \dots & v_1 \\ 0 & 0 & \dots & v_2 \\ \vdots & \vdots & \ddots & \vdots \\ 0 & 0 & \dots & v_N \end{bmatrix} \quad \right]
    \end{align*}
$$

In [83]:
class LinearLayer2D(Trainable):
    def __init__(
        self,
        dim_in: int,
        dim_out: int,
        activation: Derivable,
    ):
        self.dim_in = dim_in
        self.dim_out = dim_out
        # Add bias dim
        weight_dims = (dim_in + 1, dim_out)
        # Initialize weights
        self.weights = T.ones(
            size=weight_dims,
        )
        self.activation = activation

    # TODO If we have time we should implement the "lookup" function for the 
    # v @ W operation, since v consists mainly of zeros.
    def __call__(self, v: T.Tensor) -> T.Tensor:
        # add 1 to the last dim and fill that with 1s to create a single
        # weight matrix that contains the bias term.
        vec_plus_bias = T.cat((v, T.ones(1,)), dim=0)
        # Get y hat
        y = self.activation((vec_plus_bias.T @ self.weights))
        # This creates the nightmare tensor
        dzdw = T.zeros((self.dim_out, self.dim_in + 1, self.dim_out))
        for i in range(self.dim_out):
            dzidw = T.zeros((self.dim_in + 1, self.dim_out))
            dzidw[:, i] = vec_plus_bias
            dzdw[i] = dzidw
        # Store the derivatives on the forward pass
        self.dydv = self.activation.dydv @ self.weights.T
        self.dydw = T.sum(self.activation.dydv @ dzdw.transpose(1, 2),dim=0).T
        return y

    def update_weights(self, dnydv: T.Tensor) -> None:
        if self.dydw:
            self.weights -= dnydv @ self.dydw 
        else:
            raise RuntimeError("Perform a forward pass before trying to update weights")


test_values = [
    T.ones((10,), dtype=T.float32)
]

ll = LinearLayer2D(10, 3, Softmax())
for v in test_values:
    out = ll(v)
    dydv = ll.dydv
    dydw = ll.dydw
    w = ll.weights
    print(
        f"""Input:\n{v.numpy()},
        \nOutput:\n{out.numpy()},
        \nWeights:\n{w.numpy()},
        \nPartial derivative w.r.t v:\n{dydv.numpy()},
        \nPartial derivative w.r.t W:\n{dydw.numpy()}
        """
    )

Input:
[1. 1. 1. 1. 1. 1. 1. 1. 1. 1.],
        
Output:
[0.33333334 0.33333334 0.33333334],
        
Weights:
[[1. 1. 1.]
 [1. 1. 1.]
 [1. 1. 1.]
 [1. 1. 1.]
 [1. 1. 1.]
 [1. 1. 1.]
 [1. 1. 1.]
 [1. 1. 1.]
 [1. 1. 1.]
 [1. 1. 1.]
 [1. 1. 1.]],
        
Partial derivative w.r.t v:
[[-1.4901161e-08 -1.4901161e-08 -1.4901161e-08 -1.4901161e-08 -1.4901161e-08 -1.4901161e-08 -1.4901161e-08 -1.4901161e-08 -1.4901161e-08 -1.4901161e-08 -1.4901161e-08]
 [-1.4901161e-08 -1.4901161e-08 -1.4901161e-08 -1.4901161e-08 -1.4901161e-08 -1.4901161e-08 -1.4901161e-08 -1.4901161e-08 -1.4901161e-08 -1.4901161e-08 -1.4901161e-08]
 [-1.4901161e-08 -1.4901161e-08 -1.4901161e-08 -1.4901161e-08 -1.4901161e-08 -1.4901161e-08 -1.4901161e-08 -1.4901161e-08 -1.4901161e-08 -1.4901161e-08 -1.4901161e-08]],
        
Partial derivative w.r.t W:
[[-1.4901161e-08 -1.4901161e-08 -1.4901161e-08]
 [-1.4901161e-08 -1.4901161e-08 -1.4901161e-08]
 [-1.4901161e-08 -1.4901161e-08 -1.4901161e-08]
 [-1.4901161e-08 -1.4901161e-08

## The Model object
This object holds all teh layers and activation functions we have defined, it represents the impllementation of the full function that produces and output $\mathbf{\hat{y}}$ from an input $\mathbf{v}$.

In [84]:
class Model:
    def __init__(self, layers: list[Trainable | Derivable | object]):
        self.layers = layers

    def __call__(self, input: T.Tensor):
        """
        Performs a forward pass.
        """
        curr = input
        for l in self.layers:
            curr = l(curr)
        return curr
    
    def backpropogate(self, dLdy: T.Tensor) -> None:
        dnydv = dLdy # Start of by using the derivative of the loss w.r.t to the output
        for l in self.layers.reverse():
            # If the layer is trainable update its weights
            if l is Trainable: 
                l.update_weights(dnydv)
            # Update the derivative of the next layer output w.r.t its input 
            # (which is the derivative of the output of the current layer w.r.t to its input)
            dnydv = l.dydv


## Binary cross entropy loss
The Binary cross entropy loss function is a loss function that is primarily used to compare probability vectors (vectors where elements sum to $1$) against binary vectors (where a signle element is set to $1$ and there rest are set to $0$.) This is usally used in binary classification where a model predicts a probability of a sample having some label, for example, $cat$, $dog$, $fish$. In binary classification is only a single correct lable but to improve model performance we allow it to be kind of uncertain, so it can predict $90\%$ $dog$ , $5\%$ $cat$, $5\%$ $fish$.
$$
    L = -\big(y \log(\hat{y}) + (1 - y) \log(1 - \hat{y})\big)
$$


## Training harness
An object that trains a Model object using stochastic gradient descent (SGD)
 
```py
cel = CrossEntropyLoss()
mod = Word2Vec():
for epoch in nr_epochs:
    for b in get_batch(training_data):
        y_hat = mod(b)
        loss = cel(y_hat, y_true)
        dLdy = cel.dLdy
        mod.backpropogate(dLdy)
        ...
    ...
```

# Creating the Word2Vec model

In [85]:
INPUT_DIM = encoder.dim
OUTPUT_DIM = INPUT_DIM
LATENT_DIM = 10
word2vec = Model([
    OneHotEncoder(encoder.vocabulary), 
    LinearLayer2D(
        dim_in=INPUT_DIM,
        dim_out=LATENT_DIM,
        activation=NoActivation()
    ),
    LinearLayer2D(
        dim_in=LATENT_DIM,
        dim_out=OUTPUT_DIM,
        activation=Softmax()
    )
])

display(word2vec("banana"))
word2vec.backpropogate(1)

TypeError: eye(): argument 'n' (position 1) must be int, not Tensor

# Training, Validation, Testing

## Continous bag of words

## Skip-gram

In [ ]:
#TODO implement regularization so we can select a lambda param during validation
#TODO try different output sizes on the linear layer

# Visualizing results

## PCA
...

## UMAP
...

In [ ]:
#TODO visualize results using UMAP and PCA